## Tracking of a Fish-position dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
from ordered_set import OrderedSet
import pandas as pd
from stonesoup.types.detection import Clutter, Detection, TrueDetection
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np
from scipy.stats import uniform
# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import CovarianceMatrices, StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater
from numpy.random      import default_rng

from stonesoup.types.state import GaussianState, MarginalisedParticleState
from stonesoup.types.array import StateVectors, CovarianceMatrices
from datetime import timedelta

In [ ]:
############################################################
# 0.  Imports  +  global constants used everywhere
############################################################

# ---------- generic SMC settings ----------
num_particles          = 2000            # per-object MPF particles
P_D                    = 0.6       # detection prob.
KAPPA_Z                = 1e-5           # clutter spatial density
SEED                   = 1
rng                    = default_rng(SEED)
MAX_CLUTTER=2

# process only the first N lines of every CSV
optimise_T   = 1000
filter_T     = 1000
num_steps    = max(optimise_T, filter_T)
num_sigma2   = 1
sigma_e_factor = 0.05
sigma2_e=0.02**2


# ---------- folders & file list ----------
folder = fr"TrackedDatasets\FishData\train"
files = [
    # rf"ZebraFish-01",
    rf"ZebraFish-02",
    # rf"ZebraFish-03",
    # rf"ZebraFish-04",
    ]

############################################################
# 1–3.  Load CSV  ✧  empirical σᵉ  ✧  detections
############################################################
datasets = {}                                      # name → info-dict
priors = {}                                     # file → (lp_prior , gp_prior)

In [ ]:
name= "ZebraFish-02"
txt_path  = os.path.join(folder, name, "gt", "gt.txt")

# ------------------------------------------------------------------
# Load & keep only frame, id, 3-D position
# ------------------------------------------------------------------
cols = ["frame", "id", "x", "y", "z"]
df   = pd.read_csv(txt_path, header=None, sep=",",
                    usecols=[0, 1, 2, 3, 4], names=cols)

# timeline (1 kHz → ms indexes in MOT files)
start_time = datetime.now()
ts         = [start_time + timedelta(milliseconds=int(k))
                for k in range(num_steps)]

# ─── measurement models (one per σ²) ──────────────────────────────
meas_model = LinearGaussian(ndim_state=6, mapping=(0,2,4),
                        noise_covar=sigma2_e*np.eye(3))

# ------------------------------------------------------------------
# Build one GroundTruthPath per fish-ID
# ------------------------------------------------------------------
truths = []                                   # id → GroundTruthPath
all_true_dets=[]
max_frames=0
for fish_id, grp in df.groupby("id"):
    truth= GroundTruthPath()
    true_dets=Track()
    n_frames = len(grp)
    if n_frames>max_frames:
        max_frames=n_frames
    for k in range(num_steps):
        if k >= n_frames:
            # we’ve exhausted the real data for this fish → stop adding false poses
            break        
        row = grp.iloc[k]
        state = np.array([row.x, 0,
                          row.y, 0,
                          row.z, 0])
        truth.append(GroundTruthState(state,timestamp=ts[k]))
        true_dets.append(Detection(state_vector=np.array([row.x,row.y,row.z]),
                                    timestamp=ts[k],
                                    measurement_model=meas_model))
        
    truths.append(truth)                                # keep ordering
    all_true_dets.append(true_dets)

ts=ts[:max_frames]
print("max_frames= ",max_frames)
optimise_T,filter_T=min(optimise_T,max_frames),min(filter_T,max_frames)
# Pooled stats for clutter
all_xyz = df[["x", "y", "z"]].to_numpy(float)
x_std, y_std, z_std = np.std(all_xyz, axis=0)

# ------------------------------------------------------------------
# Build detection list:   time-indexed *set*  (True + clutter)
# Same list reused for every σ² so each grid-point sees identical data
# ------------------------------------------------------------------
all_measurements = []
all_clutter = []
for k, t in enumerate(ts):
    mset = set()
    clutter_set=set() # for plotting- need to separate clutter as will plot measurements as truths for ease
    
    # --- true detections (one per path) --------------------------
    for truth in truths:
        if np.random.rand() <= P_D:
            # assuming the data are observations rather than the groundtruth
            meas=meas_model.function(truth[k],noise=False)
            mset.add(Detection(state_vector=meas,
                                    timestamp=t,
                                    measurement_model=meas_model))

    # --- clutter -----------------------------------------------
    kappa_z = 0.5*(MAX_CLUTTER-1)/( (4*x_std)**3 )
    if MAX_CLUTTER:
        for _ in range(rng.integers(MAX_CLUTTER)):
            x = uniform.rvs(5,25, random_state=rng)
            y = uniform.rvs(5,25, random_state=rng)
            z = uniform.rvs(0,15, random_state=rng)
            mset.add(Clutter(state_vector=np.array([[x],[y],[z]]),
                            timestamp=t,
                            measurement_model=meas_model))
            clutter_set.add(Clutter(state_vector=np.array([[x],[y],[z]]),
                            timestamp=t,
                            measurement_model=meas_model))
    all_measurements.append(mset)
    all_clutter.append(clutter_set)
# ------------------------------------------------------------------
# Store everything for later
# ------------------------------------------------------------------
data = dict(
    ts            = ts,
    meas_model    = meas_model,
    measurements  = all_measurements,
    groundtruths  = truths, # list[GroundTruthPath]
    all_true_dets = all_true_dets,
    sigma2_e      = sigma2_e,
    clutter       = all_clutter
)

datasets[name] = data
print(f"⇒ Prepared {name} fish-datasets:", ", ".join(datasets))
############################################################
# 4.  Prior states  (one Marginalised-Particle + one Gaussian per file)
############################################################


# 4.  Priors  (one LP & one GP Track per truth per CSV file)
# ===============================================
priors[name]=[[],[]] #one list for lp, one for gp
for truth in truths:
    mu0 = np.array(truth[0].state_vector.flatten())
    
    # variance for positions from empirical σe; same for velocities
    Sigma0  = np.diag([sigma2_e]*6)  
    # build (d, d, N) covariance stack more succinctly
    cov_stack = np.tile(Sigma0[:, :, None], (1, 1, num_particles))
    states=multivariate_normal.rvs(mu0,Sigma0,num_particles)
    t0 = ts[0] - timedelta(milliseconds=1)    

    #  –– particle prior
    lp = Track(MarginalisedParticleState(
        state_vector = StateVectors(states.T),
        covariance   = CovarianceMatrices(cov_stack),
        weight       = np.full(num_particles, 1/num_particles),
        timestamp    = t0
    ))
    priors[name][0].append(lp)
    
    #  –– Gaussian prior
    gp = Track(GaussianState(
        state_vector = mu0,
        covar        = Sigma0,
        timestamp    = t0
    ))
    
    priors[name][1].append(gp)

In [ ]:
# 5.  Tiny helper : tolerant dictionary key look-up
############################################################
def close_key(dct, value, tol=1e-12):
    """Return the key in *dct* closest to *value* (within tol)."""
    keys = np.fromiter(dct, float)
    k    = keys[np.argmin(np.abs(keys - value))]
    if abs(k - value) > tol:
        raise KeyError(value)
    return float(k)

In [ ]:
# 6.  Cached builders  (transition-model ➜ predictor + updaters)
############################################################
from functools import lru_cache

from stonesoup.measures.state import Mahalanobis

# — Gaussian side —
from stonesoup.dataassociator.neighbour import GlobalNearestNeighbour
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.models.transition.linear import (ConstantVelocity,
    RandomWalk, OrnsteinUhlenbeck, CombinedLinearGaussianTransitionModel)
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman   import KalmanUpdater

# — Lévy / particle side —
from stonesoup.models.driver               import AlphaStableNSMDriver
from stonesoup.models.transition.levy_linear import (
    LevyRandomWalk, LevyConstantVelocity,LevyLangevin, CombinedLinearLevyTransitionModel)
from stonesoup.predictor.particle          import MarginalisedParticlePredictor
from stonesoup.updater.particle            import MarginalisedParticleUpdater
from stonesoup.resampler.particle          import SystematicResampler

RESAMPLER = SystematicResampler()           # single shared instance


@lru_cache(maxsize=None)
def build_gp(name: str, sigma_w2: float, theta:float):
    """
    Return (predictor, {σe² : updater}) for the Gaussian model attached
    to *name*.  • OU in 2-D (hidden velocity)  • RW in 2- or 3-D otherwise.
    """

    # --- choose transition model ----------------------------------
    tm = CombinedLinearGaussianTransitionModel(
                    [OrnsteinUhlenbeck(noise_diff_coeff=sigma_w2,damping_coeff=theta),
                     OrnsteinUhlenbeck(noise_diff_coeff=sigma_w2,damping_coeff=theta),
                    OrnsteinUhlenbeck(noise_diff_coeff=sigma_w2/3**2,damping_coeff=theta)])
    mdl = datasets[name]['meas_model']
    predictor = KalmanPredictor(tm)
    updater  = KalmanUpdater(mdl)
                 
    data_associator = GlobalNearestNeighbour(DistanceHypothesiser(predictor, 
                                                                  updater,
                                                                measure=Mahalanobis(), 
                                                                missed_distance=25))

    return data_associator, updater,predictor


@lru_cache(maxsize=None)
def build_lp(name: str, sigma_w2: float, theta:float, alpha: float,mu:float):
    """
    Return (predictor, {σe² : updater}) for the Lévy (MPF) model
    attached to *name*.  • Lévy-Langevin in 2-D with vel,
    • Lévy RW in 2- or 3-D otherwise.
    """
    driver = AlphaStableNSMDriver(mu_W=mu,
                                  sigma_W2=sigma_w2,
                                  c=10.0,
                                  alpha=alpha,
                                  noise_case=NoiseCase(2))
    driver_z = AlphaStableNSMDriver(mu_W=mu/3,
                                    sigma_W2=sigma_w2,
                                    c=10.0,
                                    alpha=alpha,
                                    noise_case=NoiseCase(2))
    tm = CombinedLinearLevyTransitionModel(
                    [LevyLangevin(driver=driver, noise_diff_coeff=sigma_w2,damping_coeff=theta),
                     LevyLangevin(driver=driver, noise_diff_coeff=sigma_w2,damping_coeff=theta),
                     LevyLangevin(driver=driver_z, noise_diff_coeff=sigma_w2/3**2,damping_coeff=theta)])
    
    mdl = datasets[name]['meas_model']
    predictor = MarginalisedParticlePredictor(tm)
    updater  = MarginalisedParticleUpdater(mdl, RESAMPLER)
    data_associator = GlobalNearestNeighbour(DistanceHypothesiser(predictor, 
                                                                 updater,
                                                                measure=Mahalanobis(), 
                                                                missed_distance=25))
    return data_associator, updater, predictor


In [ ]:
#Filter loglike
import copy
from scipy.optimize import minimize
from scipy.special  import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from copy import deepcopy
from pathlib import Path
from datetime import timedelta

from stonesoup.plotter import Plotterly, AnimatedPlotterly, Dimension
from stonesoup.smoother.particle import MarginalisedKalmanSmoother
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.updater.kalman      import KalmanUpdater
from stonesoup.updater.particle    import MarginalisedParticleUpdater
from stonesoup.resampler.particle  import SystematicResampler
# ────────────────────────────────────────────────────────────────
#  helper: log-likelihood for ONE file & ONE hyper-parameter set
# ────────────────────────────────────────────────────────────────
def filter_loglike(name, params, model='gp'):
    """Log p(y | θ) with σe² integrated-out (Monte-Carlo for LP)."""
    if model == 'gp':
        σw2, theta = params
        associator, updater,predictor = build_gp(name, σw2,theta)
        base_tracks     = priors[name][1]          # GP prior Tracks
    else:
        σw2, theta, α, mu = params
        associator, updater,predictor = build_lp(name, σw2,theta, α,mu)
        base_tracks     = priors[name][0]          # LP prior Tracks

    ll_total = 0.0
    all_meas = datasets[name]['measurements']       
    tracks   = [copy.deepcopy(base_track) for base_track in base_tracks]         # fresh copy

    for t, dets in enumerate(all_meas[:optimise_T]):
        timestamp=datasets[name]['ts'][t]

        hypotheses= associator.associate(tracks=tracks,
                                        detections=dets,
                                        timestamp=timestamp)

        for track in tracks:
            hypo=hypotheses[track]
            if hypo.measurement:
                det=hypo.measurement
                post       = updater.update(hypo)
                track.append(post)

                mp   = hypo.measurement_prediction
                y    = det.state_vector.flatten()
                if model == 'gp':                     # single Gaussian
                    μ = mp.state_vector.flatten()
                    Σ = mp.covar
                    diff = y - μ
                    Σinv = np.linalg.inv(Σ)
                    maha = diff @ Σinv @ diff
                    d    = y.size
                    sign, logdet = np.linalg.slogdet(Σ)
                    ll_total += -0.5*(maha + d*np.log(2*np.pi) + logdet)
                else:                                 # particle mixture
                    means = mp.state_vector           # (d,N)
                    covs  = mp.covariance             # (d,d,N)
                    d, N  = means.shape
                    ll_arr = np.empty(N)
                    for j in range(N):
                        diff  = y - means[:, j]
                        Σinv  = np.linalg.inv(covs[:, :, j])
                        maha  = diff @ Σinv @ diff
                        sign, logdet = np.linalg.slogdet(covs[:, :, j])
                        ll_arr[j] = -0.5*(maha + d*np.log(2*np.pi) + logdet)
                    ll_total += logsumexp(ll_arr) - np.log(N)
            else:
                track.append(hypo.prediction)
                ll_total += np.log(1-P_D)
        if t/100==t//100:
            print(f'measurement {t}/{optimise_T}')
    return ll_total

def filter_loglike_trackknown(name, params, model='gp'):
    """Log p(y | θ) with σe² integrated-out (Monte-Carlo for LP)."""
    if model == 'gp':
        σw2, theta = params
        associator, updater,predictor = build_gp(name, σw2,theta)
        base_tracks     = priors[name][1]          # GP prior Tracks
    else:
        σw2, theta, α, mu = params
        associator, updater,predictor = build_lp(name, σw2,theta, α,mu)
        base_tracks     = priors[name][0]          # LP prior Tracks

    ll_total = 0
    all_true_dets=datasets[name]['all_true_dets']       
    tracks   = [copy.deepcopy(base_track) for base_track in base_tracks]         # fresh copy

    for i, track in enumerate(tracks):
        truth_dets=all_true_dets[i]
        ll_total-= np.log(optimise_T)
        for t, det in enumerate(truth_dets[:optimise_T]):
            timestamp=datasets[name]['ts'][t]
            pred  = predictor.predict(track[t-1], timestamp=det.timestamp)
            hypo  = SingleHypothesis(pred, det)
            post  = updater.update(hypo)
            track.append(post)

            mp   = hypo.measurement_prediction
            y    = det.state_vector.flatten()
            if model == 'gp':                     # single Gaussian
                μ = mp.state_vector.flatten()
                Σ = mp.covar
                diff = y - μ
                Σinv = np.linalg.inv(Σ)
                maha = diff @ Σinv @ diff
                d    = y.size
                sign, logdet = np.linalg.slogdet(Σ)
                ll_total += -0.5*(maha + d*np.log(2*np.pi) + logdet)
            else:                                 # particle mixture
                means = mp.state_vector           # (d,N)
                covs  = mp.covariance             # (d,d,N)
                d, N  = means.shape
                ll_arr = np.empty(N)
                for j in range(N):
                    diff  = y - means[:, j]
                    Σinv  = np.linalg.inv(covs[:, :, j])
                    maha  = diff @ Σinv @ diff
                    sign, logdet = np.linalg.slogdet(covs[:, :, j])
                    ll_arr[j] = -0.5*(maha + d*np.log(2*np.pi) + logdet)
                ll_total += logsumexp(ll_arr) - np.log(N)
            if t/100==t//100:
                print(f'measurement {t}/{optimise_T} for fish {i}')
    return ll_total



In [ ]:
def _pretty(fig, xlab, ylab):
        fig.update_layout(width=1200, height=640, plot_bgcolor="white",
                        xaxis=dict(showgrid=True, gridcolor="lightgray",
                                    title=dict(text=xlab, font=dict(size=20))),
                        yaxis=dict(showgrid=True, gridcolor="lightgray",
                                    title=dict(text=ylab, font=dict(size=20))),
                        legend=dict(font=dict(size=15),
                                    bordercolor="Black", borderwidth=2,
                                    orientation='h',y=-0.15))

In [ ]:
from stonesoup.plotter import Plotterly, Dimension

xyplotter = Plotterly(autosize=False, width=1500, height=800)
colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']
mapping=[0,2]
# observed mid / bid / ask
for i,truth in enumerate(truths):
    color = colors[i]
    xyplotter.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                line=dict(width=3,color=color,dash='dash'))
xyplotter.plot_measurements(all_clutter, mapping,marker=dict(size=8))

_pretty(xyplotter.fig, "x-position","y-position")
xyplotter.fig.show()

In [ ]:
from stonesoup.plotter import Plotterly, Dimension

xyplotter1d = Plotterly(autosize=False, width=1500, height=800,dimension=Dimension.ONE)
colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']
mapping=[0]
# observed mid / bid / ask
for i,truth in enumerate(truths):
    color = colors[i]
    xyplotter1d.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                line=dict(width=3,color=color,dash='dash'))
xyplotter1d.plot_measurements(all_clutter, mapping,marker=dict(size=8))

_pretty(xyplotter1d.fig, "time","x")
xyplotter1d.fig.show()

In [ ]:
xzplotter = Plotterly(autosize=False, width=1500, height=800)
mapping=[0,4]
# observed mid / bid / ask
for i,truth in enumerate(truths):
    color = colors[i]
    xzplotter.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                line=dict(width=3,color=color,dash='dash'))
xzplotter.plot_measurements(all_clutter, mapping,marker=dict(size=8))

_pretty(xzplotter.fig, "x-position","z-position")
xzplotter.fig.show()

In [ ]:
yzplotter = Plotterly(autosize=False, width=1500, height=800)
mapping=[2,4]

for i,truth in enumerate(truths):
    color = colors[i]
    yzplotter.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                line=dict(width=3,color=color,dash='dash'))
yzplotter.plot_measurements(all_clutter, mapping,marker=dict(size=8))

_pretty(yzplotter.fig, "y-position","z-position")
yzplotter.fig.show()

In [ ]:
def obj_gp(x):
    sigw2 = np.exp(x[0])
    theta = x[1]             
    print(np.sqrt(sigw2),theta)
    return -filter_loglike(name,(sigw2,theta), model='gp')

def obj_lp(x):
    sigw2 = np.exp(x[0])
    theta = x[1]               
    alpha = x[2]
    mu    = x[3]
    if np.abs(alpha-1.0)<0.01:
        if np.random.random()<0.5:
            alpha=0.95
        else:
            alpha=1.05
    print(sigw2,theta,alpha,mu)
    return -filter_loglike(name,(sigw2,theta, alpha,mu), model='lp')

In [ ]:
sigma_w2_gp=5.4e6 #1029**2 
theta_gp= 68

sigma_w2_lp= 5.4e14
theta_lp= 50
alpha= 0.3
mu   = 0

In [ ]:
# 4b.  Likelihood:
# ------------------------------------------------------------------
gp_l=filter_loglike(name,(sigma_w2_gp,theta_gp),'gp')
print(f'gp likelihood= {gp_l}')
lp_l=filter_loglike(name,(sigma_w2_lp,theta_lp,alpha,mu),'lp')
print(f'lp likelihood= {lp_l}')

In [ ]:
tracks_known=False
if tracks_known:
    multiobjlabel='tracksknown'
else:
    multiobjlabel='multiobject'
if MAX_CLUTTER>1:
    clutterlabel='clutter'
else:
    clutterlabel='noclutter'

# need clutter and multi, then no clutter multi, then no clutter no multi

In [ ]:
out_root = Path(r"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB"
                rf"\PROJECT- Implementation of N-G TAs in SS framework\RandomFishPlots\{clutterlabel}_{multiobjlabel}")
out_root.mkdir(parents=True, exist_ok=True)

def _pretty(fig, xlab, ylab):
    fig.update_layout(width=1200, height=640, plot_bgcolor="white",
                    xaxis=dict(showgrid=True, gridcolor="lightgray",
                                title=dict(text=xlab, font=dict(size=20))),
                    yaxis=dict(showgrid=True, gridcolor="lightgray",
                                title=dict(text=ylab, font=dict(size=20))),
                    legend=dict(font=dict(size=15),
                                bordercolor="Black", borderwidth=2))
gp_associator , gp_updater, gp_predictor= build_gp(name,sigma_w2_gp,theta_gp)    # <- SINGLE object, not a dict
lp_associator, lp_updater, lp_predictor= build_lp(name, sigma_w2_lp,theta_lp,alpha,mu)

data = datasets[name]
truths = data['groundtruths']
all_true_dets=data['all_true_dets']       
num_truths=len(truths)
lp_priors=priors[name][0]
gp_priors=priors[name][1]
LP_tracks = [Track(copy.copy(lp_prior[0])) for lp_prior in lp_priors]
GP_tracks = [Track(copy.copy(gp_prior[0]))for gp_prior in gp_priors]

if tracks_known: 
    all_true_dets=datasets[name]['all_true_dets']
    for k, true_dets in enumerate(all_true_dets):
        LP_track=LP_tracks[k]
        GP_track=GP_tracks[k]

        for t, meas in enumerate(true_dets):
            # ---- Lévy (MPF) update ---------------------------------------
            lp_pred = lp_predictor.predict(LP_track[-1], timestamp=meas.timestamp)
            lp_hypo = SingleHypothesis(lp_pred, meas)
            lp_post = lp_updater.update(lp_hypo)
            LP_track.append(lp_post)

            # ---- Gaussian (Kalman) update --------------------------------
            gp_pred = gp_predictor.predict(GP_track[-1], timestamp=meas.timestamp)
            gp_hypo = SingleHypothesis(gp_pred, meas)
            gp_post = gp_updater.update(gp_hypo)
            GP_track.append(gp_post)
            if t/100==t//100:
                print(f'measurement {t}/{filter_T} for fish {k}')
else:
    
    for t, dets in enumerate(all_measurements[:filter_T]):
        timestamp=ts[t]

        lp_hypotheses=lp_associator.associate(tracks=LP_tracks,
                                            detections=dets,
                                            timestamp=timestamp)
        for k,lp_track in enumerate(LP_tracks):
            # Lévy
            lp_hypo= lp_hypotheses[lp_track]
            if lp_hypo.measurement:
                lp_post       = lp_updater.update(lp_hypo)
                lp_track.append(lp_post)
            else:
                lp_track.append(lp_hypo.prediction)
            if t/50==t//50:
                print(f'lp measurement {t}/{filter_T} for fish {k}')
        gp_hypotheses=gp_associator.associate(tracks=GP_tracks,
                                            detections=dets,
                                            timestamp=timestamp)
        for k,gp_track in enumerate(GP_tracks):
            # Gaussian
            gp_hypo= gp_hypotheses[gp_track]
            if gp_hypo.measurement:
                gp_post       = gp_updater.update(gp_hypo)
                gp_track.append(gp_post)
            else:
                gp_track.append(gp_hypo.prediction)
            
            if t/50==t//50:
                print(f'gp measurement {t}/{filter_T} for fish {k}')
    print(name,"track filtered")

In [ ]:
# # 9.  Nelder–Mead optimisation
# ###########################################################
# # print("\n=== Gaussian RW optimisation ===")
# # res_gp = minimize(obj_gp,
# #                   x0=[np.log(sigma_w2_gp),theta_gp],
# #                   method='Nelder-Mead',
# #                   options={'maxiter': 1000, 'disp': True},
# #                   bounds=[(None,None),(None,None)])

# # sigma_gp_opt, theta_gp_opt= (np.exp(res_gp.x[0]/2),
# #                              res_gp.x[1])
# # print("Best σ_w (GP):", sigma_gp_opt, "best theta:", theta_gp_opt)


# print("\n=== Lévy RW optimisation ===")
# res_lp = minimize(obj_lp,
#                   x0=[np.log(sigma_w2_lp),theta_lp,alpha,mu],
#                   method='Nelder-Mead',
#                   options={'maxiter': 1000, 'disp': True},
#                            bounds=[(None,None),(None,None),(0.05,1.95),(-0.00,0.00)])

# (sigma_lp_opt, theta_lp_opt,alpha_opt,mu_opt)= (np.exp(res_lp.x[0]/2),
#                                           res_lp.x[1], 
#                                           res_lp.x[2],
#                                           res_lp.x[3])

# print("Best σ_w (LP):", sigma_lp_opt,"best theta:",theta_lp_opt,"best alpha:" ,alpha_opt, "best mu",mu_opt)

In [ ]:
mappings={'xy':[0,2],
        'xz':[0,4],
        'yz':[2,4]
}
times = ts
colors=['#00CC96','#FFA15A','#B6E880','#AB63FA','red']

for axes, mapping in mappings.items():
    print(axes,mapping)
    xyplotter = Plotterly(autosize=False, width=1500, height=800)
    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_tracks=[]

    xyplotter.plot_measurements(all_clutter, mapping, label="Clutter",marker=dict(size=8))
    for i, truth in enumerate(truths):
        color = colors[i]
        LP_tracks[i]=Track(LP_tracks[i][1:]) #[1:] means we exclude prior state (isn't a prediction or a post)
        GP_tracks[i]=Track(GP_tracks[i][1:])
        xyplotter.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                    line=dict(width=1,color=color,dash='dash'))
        xyplotter.plot_tracks(GP_tracks[i], mapping, track_label=f"Gaussian {i}", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color,dash="dot"))
        xyplotter.plot_tracks(LP_tracks[i], mapping, track_label=f"Lévy {i}", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color,dash='solid'))
        RTStrack=RTSsmoother.smooth(LP_tracks[i]) 
        RTS_tracks.append(RTStrack)
        xyplotter.plot_tracks(RTS_tracks[i], mapping, track_label=f"Lévy {i} RTS", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color, dash='longdashdot'))
        
    _pretty(xyplotter.fig,f"{axes[0]}-position",f"{axes[1]}-position")
    # xyplotter.fig.show()
    xyplotter.fig.write_html(out_root / f"{name}_2D{axes}pos.html")
    print("🎉  Filtering complete – HTML plots written to", out_root)


    # # -----------------------------------------------------------------
    # # 4-B.  2-D animated trajectory (skip if pure-1D data)
    # # -----------------------------------------------------------------

    animatedplotter = AnimatedPlotterly(ts)
    animatedplotter.plot_measurements(all_clutter, mapping, label="Clutter",marker=dict(size=8))
    for i, truth in enumerate(truths):
        color = colors[i]
        animatedplotter.plot_ground_truths(truth, mapping, truths_label=f"Fish {i} gt",
                    line=dict(width=1,color=color,dash='dash'))
        animatedplotter.plot_tracks(GP_tracks[i], mapping, track_label=f"Gaussian {i}", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color,dash="dot"))
        animatedplotter.plot_tracks(LP_tracks[i], mapping, track_label=f"Lévy {i}", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color,dash='solid'))    
        animatedplotter.plot_tracks(RTS_tracks[i], mapping, track_label=f"Lévy {i} RTS", uncertainty=False,
                    mode='lines',line=dict(width=1, color=color, dash='longdashdot'))
        
    _pretty(animatedplotter.fig, f"{axes[0]}-position",f"{axes[1]}-position")
    # animatedplotter.fig.write_html(out_root / f"{name}_2D{axes}animatedpos.html")
    print("🎉  Filtering complete – HTML plots written to", out_root)

oneD_axes=['x','xvel','y','yvel','z','zvel']
for mapping, axis in enumerate(oneD_axes):
    print(axis,mapping)
    # -----------------------------------------------------------------
    # 4-A.  1-D x-coordinate panel
    # -----------------------------------------------------------------
    p1 = Plotterly(dimension=Dimension.ONE)
    mapping=[mapping]
    if mapping in [[0],[2],[4]]:
        p1.plot_measurements(all_clutter, mapping, label="Clutter",marker=dict(size=8))
    for i, truth in enumerate(truths):
        color = colors[i]
        p1.plot_tracks(GP_tracks[i], mapping, track_label=f"Gaussian {i}", uncertainty=True,
                    mode='lines',line=dict(width=1, color=color,dash="dot"))
        p1.plot_tracks(LP_tracks[i], mapping, track_label=f"Lévy {i}", uncertainty=True,
                    mode='lines',line=dict(width=1, color=color,dash='solid'))
        p1.plot_tracks(RTS_tracks[i], mapping, track_label=f"Lévy {i} RTS", uncertainty=True,
                    mode='lines',line=dict(width=1, color=color, dash='longdashdot'))
        
    _pretty(p1.fig, f"{axis}","time")
    p1.fig.write_html(out_root / f"{name}_1D{axis}.html")
    print("🎉  Filtering complete – HTML plots written to", out_root)
    # p1.fig.show()
